# Limpieza de datos con criterio — Superstore


## 1 — Tipos correctos

Antes de cualquier análisis, confirmar que cada columna tiene el tipo correcto.
Un `object` donde debería haber `datetime` o `float` rompe filtros, agrupaciones y gráficos.


In [1]:
import pandas as pd

df = pd.DataFrame({
    "id_cliente": [1, 2, 3, 4, 5],
    "nombre":     ["Ana", "Carlos", "Marta", None, "Pedro"],
    "edad":       ["32", "28", "none", "41", "36"],
    "importe":    [899.0, None, 349.0, 45.0, 25.0],
    "fecha":      ["2024-01-15", "2024-02-20", "2024-03-01", "15/03/2024", "2024-04-10"],
    "ciudad":     ["Madrid", "Barcelona", "madrid", "Sevilla", "Valencia"]
})

# "none" como string no es nulo para pandas — hay que convertirlo primero
df["edad"] = df["edad"].replace("none", pd.NA)
df["edad"] = pd.to_numeric(df["edad"])

# format='mixed' acepta mezcla de formatos en la misma columna
# dayfirst=True indica que el día va antes que el mes (15/03/2024)
df["fecha"] = pd.to_datetime(df["fecha"], format="mixed", dayfirst=True)

print(df.dtypes)
print("----------------")
print(df.head())

id_cliente             int64
nombre                   str
edad                 float64
importe              float64
fecha         datetime64[us]
ciudad                   str
dtype: object
----------------
   id_cliente  nombre  edad  importe      fecha     ciudad
0           1     Ana  32.0    899.0 2024-01-15     Madrid
1           2  Carlos  28.0      NaN 2024-02-20  Barcelona
2           3   Marta   NaN    349.0 2024-01-03     madrid
3           4     NaN  41.0     45.0 2024-03-15    Sevilla
4           5   Pedro  36.0     25.0 2024-10-04   Valencia


In [2]:
df["fecha"] = pd.to_datetime(df["fecha"])
print(df.dtypes)
print("--------------")


id_cliente             int64
nombre                   str
edad                 float64
importe              float64
fecha         datetime64[us]
ciudad                   str
dtype: object
--------------


## 2 — Nulos: detectar y decidir

Tres decisiones posibles por columna: `fillna`, `dropna`, o dejar como está.
La decisión depende de si el valor ausente es recuperable o crítico.


In [3]:
print(df.isnull().sum())

id_cliente    0
nombre        1
edad          1
importe       1
fecha         0
ciudad        0
dtype: int64


### nombre → `fillna('Desconocido')`

El registro existe, solo falta el nombre. Rellenar no distorsiona el análisis.


In [4]:
df["nombre"] = df["nombre"].fillna("Desconocido")
print(df['nombre'])

0            Ana
1         Carlos
2          Marta
3    Desconocido
4          Pedro
Name: nombre, dtype: str


### edad → `fillna(mediana)`

Variable numérica: mediana porque no es sensible a outliers como la media.


In [5]:
mediana_edad = df["edad"].median()
df["edad"] = df["edad"].fillna(mediana_edad).astype(int)
print(f"Mediana usada para relleno: {mediana_edad}")
print(df['edad'])

Mediana usada para relleno: 34.0
0    32
1    28
2    34
3    41
4    36
Name: edad, dtype: int64


### importe → `dropna`

Dato crítico. Un importe desconocido no se rellena — se elimina la fila.


In [6]:
print(f"Filas antes:  {len(df)}")
df = df.dropna(subset=["importe"])
print(f"Filas después: {len(df)}")

Filas antes:  5
Filas después: 4


## 3 — Columnas con más del 60% de nulos

Por encima de ese umbral la columna no aporta — se evalúa si merece ser eliminada.


In [7]:
umbral_nulos = 0.5  # variable de configuración al inicio, no hardcodeada

columnas_con_demasiados_nulos = [
    col for col in df.columns
    if df[col].isnull().mean() > umbral_nulos
]

print(f"Columnas a eliminar: {columnas_con_demasiados_nulos}")
df = df.drop(columns=columnas_con_demasiados_nulos)

Columnas a eliminar: []


## 4 — Inconsistencias de formato

`'Madrid'` y `'madrid'` son valores distintos para pandas. `.str.strip().str.title()` unifica.


In [8]:
print("Antes:")
print(df["ciudad"].value_counts())

df["ciudad"] = df["ciudad"].str.strip().str.title()

print("\nDespués:")
print(df["ciudad"].value_counts())

Antes:
ciudad
Madrid      1
madrid      1
Sevilla     1
Valencia    1
Name: count, dtype: int64

Después:
ciudad
Madrid      2
Sevilla     1
Valencia    1
Name: count, dtype: int64


## 5 — Documentar las decisiones

Cada decisión de limpieza va comentada: qué se hizo y por qué, no solo el código.


In [9]:
# DECISIONES DE LIMPIEZA — 2024-06-22

# nombre: rellenado con 'Desconocido' porque el cliente tiene
# transacciones asociadas y eliminar la fila haría perder revenue real

# edad: rellenada con mediana (34) en vez de media porque
# la distribución puede tener outliers en edades extremas

# importe: fila eliminada porque no se puede imputar el valor
# de una transacción financiera sin dato real

# ciudad: normalizada con str.title() para evitar grupos
# duplicados en groupby (madrid != Madrid para pandas)

print(df)

   id_cliente       nombre  edad  importe      fecha    ciudad
0           1          Ana    32    899.0 2024-01-15    Madrid
2           3        Marta    34    349.0 2024-01-03    Madrid
3           4  Desconocido    41     45.0 2024-03-15   Sevilla
4           5        Pedro    36     25.0 2024-10-04  Valencia


## 6 — Función de limpieza reutilizable

Si el mismo pipeline se aplica a varios datasets, se encapsula en una función.


In [10]:
def limpiar_dataset(df, umbral_nulos=0.5, columnas_criticas=None):
    df = df.copy()

    # Eliminar columnas con demasiados nulos
    cols_a_eliminar = [c for c in df.columns if df[c].isnull().mean() > umbral_nulos]
    df = df.drop(columns=cols_a_eliminar)
    print(f'Columnas eliminadas por nulos: {cols_a_eliminar}')

    # Eliminar filas con nulos en columnas críticas
    if columnas_criticas:
        antes = len(df)
        df = df.dropna(subset=columnas_criticas)
        print(f'Filas eliminadas por nulos críticos: {antes - len(df)}')

    # Eliminar duplicados
    antes = len(df)
    df = df.drop_duplicates()
    print(f'Duplicados eliminados: {antes - len(df)}')

    return df


# Reconstruir el DataFrame original para demostrar la función desde cero
df_raw = pd.DataFrame({
    "id_cliente": [1, 2, 3, 4, 5],
    "nombre":     ["Ana", "Carlos", "Marta", None, "Pedro"],
    "edad":       ["32", "28", "none", "41", "36"],
    "importe":    [899.0, None, 349.0, 45.0, 25.0],
    "fecha":      ["2024-01-15", "2024-02-20", "2024-03-01", "15/03/2024", "2024-04-10"],
    "ciudad":     ["Madrid", "Barcelona", "madrid", "Sevilla", "Valencia"]
})

df_limpio = limpiar_dataset(df_raw, umbral_nulos=0.5, columnas_criticas=["importe"])
print(df_limpio)

Columnas eliminadas por nulos: []
Filas eliminadas por nulos críticos: 1
Duplicados eliminados: 0
   id_cliente nombre  edad  importe       fecha    ciudad
0           1    Ana    32    899.0  2024-01-15    Madrid
2           3  Marta  none    349.0  2024-03-01    madrid
3           4    NaN    41     45.0  15/03/2024   Sevilla
4           5  Pedro    36     25.0  2024-04-10  Valencia


In [11]:
#df = df.copy()
df.dtypes

id_cliente             int64
nombre                   str
edad                   int64
importe              float64
fecha         datetime64[us]
ciudad                   str
dtype: object